# Preparação dos dados


Juntar 2 datasets

## Carregando os dados

In [1]:
from pathlib import Path
import re
import pandas as pd

# funciona rodando da raiz do repositório ou de dentro de entrega-2
raiz = Path.cwd()
if not (raiz / "data").exists():
    raiz = raiz.parent

ditadura = pd.read_csv(raiz / "data" / "musicas-ditadura.csv")
outros = pd.read_csv(raiz / "data" / "mpb_adicional_completo_filtrado.csv")

print("ditadura:", ditadura.shape)
print("outros:", outros.shape)

ditadura: (503, 11)
outros: (470, 12)


## Ano e grupo

Pega anos de lançamento e tira músicas sem letra

In [2]:
ditadura["ano"] = ditadura["release_date"].astype(str).str[:4].astype(int)
ditadura["grupo"] = "ditadura"

outros["ano"] = outros["release_year"].astype(int)
outros["grupo"] = "outros"

sem_letra = outros["lyrics"].isna().sum()
outros = outros.dropna(subset=["lyrics"])
print("outros sem letra:", sem_letra)

outros sem letra: 18


## Duplicatas

Tira músicas duplicadas

In [3]:
def chave(titulo, artista):
    t = re.sub(r"\(.*?\)|-.*$", "", str(titulo).lower())
    t = re.sub(r"\W+", " ", t).strip()
    return t + "|" + str(artista).lower()

def limpar(df):
    df = df.copy()
    df["chave"] = [chave(t, a) for t, a in zip(df["track_name"], df["artist_name"])]
    df["ao_vivo"] = df["track_name"].astype(str).str.contains("ao vivo|live|remaster", case=False)
    df = df.sort_values("ao_vivo", kind="stable")
    return df.drop_duplicates("chave").drop_duplicates("lyrics").sort_index()

chaves_ditadura = {chave(t, a) for t, a in zip(ditadura["track_name"], ditadura["artist_name"])}

ditadura_limpa = limpar(ditadura)
outros_limpa = limpar(outros)
print("duplicatas removidas (ditadura):", len(ditadura) - len(ditadura_limpa))
print("duplicatas removidas (outros):", len(outros) - len(outros_limpa))

antes = len(outros_limpa)
outros_limpa = outros_limpa[~outros_limpa["chave"].isin(chaves_ditadura)]
print("regravações removidas de outros:", antes - len(outros_limpa))

duplicatas removidas (ditadura): 7
duplicatas removidas (outros): 68
regravações removidas de outros: 17


## Lista de nós

In [4]:
colunas = ["track_name", "artist_name", "album_name", "ano", "grupo", "lyrics"]
nos = pd.concat([ditadura_limpa[colunas], outros_limpa[colunas]], ignore_index=True)
nos.insert(0, "no_id", range(len(nos)))
nos["n_palavras"] = nos["lyrics"].str.split().str.len()

nos.to_csv(raiz / "entrega-2" / "nos.csv", index=False)
print("total de nós:", len(nos))

total de nós: 863


## Resumo

In [5]:
print(nos["grupo"].value_counts().to_string())
print()
print(nos.groupby("grupo")["ano"].agg(["min", "max"]).to_string())
print()
print(nos.groupby("grupo")["n_palavras"].agg(["mean", "median"]).round(0).to_string())

grupo
ditadura    496
outros      367

           min   max
grupo               
ditadura  1964  1985
outros    1963  2026

           mean  median
grupo                  
ditadura  180.0   168.0
outros    214.0   194.0


In [6]:
for g in ["ditadura", "outros"]:
    print("artistas com mais músicas em", g)
    print(nos[nos["grupo"] == g]["artist_name"].value_counts().head(8).to_string())
    print()

artistas com mais músicas em ditadura
artist_name
Ney Matogrosso                   58
Os Mutantes                      55
Rita Lee                         50
Raul Seixas                      38
Chico Buarque                    36
Rita Lee, Roberto De Carvalho    35
Gal Costa                        20
Caetano Veloso                   17

artistas com mais músicas em outros
artist_name
Legião Urbana              25
Marisa Monte               23
Os Paralamas Do Sucesso    20
Kid Abelha                 19
Chico César                16
Raul Seixas                11
Bezerra Da Silva           11
Roberto Carlos             10



In [7]:
nos[["no_id", "track_name", "artist_name", "ano", "grupo", "n_palavras"]].head(10)

,no_id,track_name,artist_name,ano,grupo,n_palavras
0,0,Mágica,Os Mutantes,1969,ditadura,150
1,1,Rico Sem Dinheiro,Evinha,1971,ditadura,166
2,2,Que Loucura,"Rita Lee, Roberto De Carvalho",1978,ditadura,162
3,3,1964 II,Ney Matogrosso,1975,ditadura,153
4,4,Flagra,"Rita Lee, Roberto De Carvalho",1982,ditadura,127
5,5,Apesar de Você,Chico Buarque,1970,ditadura,273
6,6,Prisioneira Do Amor,Rita Lee,1970,ditadura,175
7,7,Sampa,Caetano Veloso,1978,ditadura,231
8,8,O Que Foi Feito Devera De Vera,"Elis Regina, Milton Nascimento",1978,ditadura,86
9,9,Baila Comigo,"Rita Lee, Roberto De Carvalho",1980,ditadura,323
